# Imports

In [1]:
import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
import tqdm.notebook as tqdm
import matplotlib.pyplot as plt

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except:
        pass

# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [3]:
class Tokenizer(transformers.GPT2Tokenizer):

    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        if token_ids_1 is None:
            return [self.bos_token_id, *token_ids_0, self.eos_token_id]

        return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]

In [4]:
tokenizer = Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'Tokenizer'.


Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

# Осмотр датасета

In [5]:
import pandas as pd

df = pd.read_csv(
    "pare.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="skip",
    header=None
)

df.columns = ["id_ru", "text_ru", "id_en", "text_en"]

print("Размер:", df.shape)
print(df.head())

Размер: (779067, 4)
   id_ru                                            text_ru   id_en  \
0    243  Один раз в жизни я делаю хорошее дело... И оно...    3257   
1   5409                      Давайте что-нибудь попробуем!    1276   
2   5410                               Мне пора идти спать.    1277   
3   5411                                    Что ты делаешь?   16492   
4   5411                                    Что ты делаешь?  511884   

                                             text_en  
0  For once in my life I'm doing a good deed... A...  
1                               Let's try something.  
2                             I have to go to sleep.  
3                                What are you doing?  
4                                  What do you make?  


In [ ]:
# print(df.info())
# print(df.head())

# Подготовка данных к модели (tokenization + teacher forcing)

In [7]:
import tensorflow as tf
import numpy as np

MAX_LEN = 50
BATCH = 64

def encode_pair(ru, en):
    input_tokens = tokenizer.encode(ru, add_special_tokens=True, truncation=True, max_length=MAX_LEN)
    output_tokens = tokenizer.encode(en, add_special_tokens=True, truncation=True, max_length=MAX_LEN)
    decoder_input = output_tokens[:-1]
    decoder_target = output_tokens[1:]
    mask = [1] * len(decoder_target)
    return (input_tokens, decoder_input), (decoder_target, mask)

def pad_to_max(x, max_len=MAX_LEN):
    if len(x) >= max_len:
        return x[:max_len]
    return x + [tokenizer.pad_token_id] * (max_len - len(x))

def make_dataset(df, batch_size=BATCH):
    enc_in, dec_in, dec_out, masks = [], [], [], []
    for _, row in df.iterrows():
        (inp, dec_inp), (target, mask) = encode_pair(row["text_ru"], row["text_en"])
        enc_in.append(pad_to_max(inp))
        dec_in.append(pad_to_max(dec_inp))
        dec_out.append(pad_to_max(target))
        masks.append(pad_to_max(mask))
    enc_in = np.array(enc_in, dtype=np.int32)
    dec_in = np.array(dec_in, dtype=np.int32)
    dec_out = np.array(dec_out, dtype=np.int32)
    masks = np.array(masks, dtype=np.float32)  # sample_weight
    ds = tf.data.Dataset.from_tensor_slices(((enc_in, dec_in), dec_out, masks))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [ ]:
# def get_model(
#     units: int,
#     n_tokens: int,
#     n_labels: int,
#     n_stacks: int = 1,
#     bidirectional: bool = False,
#     name: str | None = None,
#     cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
# ) -> keras.Model:
#     '''Creates a model with RNN architecture for sequence to sequence classification.

#     Arguments:
#         units: dimensionality of RNN cells
#         n_tokens: number of tokens in the tokenizer dictionary
#         n_labels: number of labels to be predicted
#         n_stacks: number of RNN cells in the stack (1 -- no stacking)
#         bidirectional: whether or not the model is bidirectional
#         name: the model name
#         cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

#     Returns:
#         The model'''
#     ...

In [8]:
from keras import layers, Model

def get_model(units, n_tokens, n_labels, name=None):
    encoder_inputs = layers.Input(shape=(MAX_LEN,), dtype="int32", name="encoder_inputs")
    decoder_inputs = layers.Input(shape=(MAX_LEN,), dtype="int32", name="decoder_inputs")

    # Отключаем автоматическое mask propagation
    embedding = layers.Embedding(n_tokens, units, mask_zero=False, name="token_embedding")
    enc_emb = embedding(encoder_inputs)
    dec_emb = embedding(decoder_inputs)

    # Энкодер
    encoder_lstm = layers.LSTM(units, return_sequences=True, return_state=True, name="encoder_lstm")
    enc_out, enc_h, enc_c = encoder_lstm(enc_emb)

    # Декодер
    decoder_lstm = layers.LSTM(units, return_sequences=True, return_state=True, name="decoder_lstm")
    dec_out_seq, _, _ = decoder_lstm(dec_emb, initial_state=[enc_h, enc_c])

    # Attention (ВАЖНО: создаём слой отдельно, но не позволяем Keras пробрасывать маску)
    attn = layers.AdditiveAttention(name="additive_attn")

    # вызываем вручную — без передачи масок!
    context = attn([dec_out_seq, enc_out], use_causal_mask=False)  

    # Объединяем контекст и выход декодера
    concat = layers.Concatenate(axis=-1)([dec_out_seq, context])
    logits = layers.TimeDistributed(layers.Dense(n_labels, activation="softmax"), name="time_dist_logits")(concat)

    model = Model([encoder_inputs, decoder_inputs], logits, name=name or "seq2seq_attn")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [14]:
model = get_model(
    units=256,
    n_tokens=len(tokenizer),
    n_labels=len(tokenizer)
)

In [15]:

# model = get_model(
#     units=256,
#     n_tokens=len(tokenizer),
#     n_labels=len(tokenizer),
#     n_stacks=1,
#     bidirectional=False
# )
train_ds = make_dataset(df, batch_size=32)

In [16]:
model.summary()

Model: "seq2seq_attn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ decoder_inputs (InputLayer)   │ (None, 45)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_inputs (InputLayer)   │ (None, 45)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding (Embedding)   │ (None, 45, 256)           │      12,865,792 │ encoder_inputs[0][0],      │
│                               │                           │                 │ decoder_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_lstm (LSTM)           │ [(None, 45, 256), (None,  │         525,312 │ token_embedding[0][0]      │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_lstm (LSTM)           │ [(None, 45, 256), (None,  │         525,312 │ token_embedding[1][0],     │
│                               │ 256), (None, 256)]        │                 │ encoder_lstm[0][1],        │
│                               │                           │                 │ encoder_lstm[0][2]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ additive_attn                 │ (None, 45, 256)           │             256 │ decoder_lstm[0][0],        │
│ (AdditiveAttention)           │                           │                 │ encoder_lstm[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_1 (Concatenate)   │ (None, 45, 512)           │               0 │ decoder_lstm[0][0],        │
│                               │                           │                 │ additive_attn[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ time_dist_logits              │ (None, 45, 50257)         │      25,781,841 │ concatenate_1[0][0]        │
│ (TimeDistributed)             │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 39,698,513 (151.44 MB)

 Trainable params: 39,698,513 (151.44 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
import os
os.makedirs("models_attention", exist_ok=True)

In [18]:
from keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath=os.path.join("models_attention", "best_model.keras"),
    monitor="loss",          # можно заменить на 'val_loss', если будет валидация
    save_best_only=True,     # сохраняем только лучшую модель
    save_weights_only=False, # сохраняем всю модель
    mode="min",              # минимизация loss
    verbose=1
)

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

# Training

In [19]:
history = model.fit(
    train_ds,
    epochs=1,
    callbacks=[checkpoint]
)

24346/24346 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8421 - loss: 1581.6466
Epoch 1: loss improved from None to 360.30420, saving model to models_attention\best_model.keras
24346/24346 ━━━━━━━━━━━━━━━━━━━━ 35219s 1s/step - accuracy: 0.8637 - loss: 360.3042


Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [20]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    model: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text`translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''
    ...

In [ ]:
# def translate(
#     text: str,
#     tokenizer: Tokenizer,
#     model: keras.Model,
#     max_len: int = 20
# ) -> str:
#     input_tokens = tokenizer.encode(text, add_special_tokens=True)
#     input_tensor = tf.constant([input_tokens])
#     output_tokens = [tokenizer.bos_token_id]

#     for _ in range(max_len):
#         pred = model.predict([input_tensor, tf.constant([output_tokens])], verbose=0)
#         next_token = int(np.argmax(pred[0, -1]))
#         if next_token == tokenizer.eos_token_id:
#             break
#         output_tokens.append(next_token)

#     return tokenizer.decode(output_tokens[1:])

In [21]:
def translate(text: str, tokenizer: Tokenizer, model: keras.Model, max_len: int = 20):
    """
    Предполагается что `model` обучен и имеет слои с именами:
      - 'token_embedding', 'encoder_lstm', 'decoder_lstm', 'additive_attn', 'time_dist_logits'
    Мы будем вызывать LSTM слоя по 1 шагу (timesteps=1) для декодера.
    """
    # 1) подготовка входа
    enc_tokens = tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=MAX_LEN)
    enc_tokens = pad_to_max(enc_tokens, MAX_LEN)
    enc_tokens = np.array([enc_tokens], dtype=np.int32)  # batch=1

    # Получаем слои
    embedding = model.get_layer("token_embedding")
    encoder_lstm = model.get_layer("encoder_lstm")
    decoder_lstm = model.get_layer("decoder_lstm")
    attn_layer = model.get_layer("additive_attn")
    logits_layer = model.get_layer("time_dist_logits")  # TimeDistributed(Dense)

    # 2) прогоним энкодер (получим enc_out и состояния)
    enc_emb = embedding(enc_tokens)  # (1, L, units)
    enc_out, enc_h, enc_c = encoder_lstm(enc_emb)

    # 3) начинаем декодировать шаг за шагом
    cur_token = np.array([[tokenizer.bos_token_id]], dtype=np.int32)  # начнём с BOS
    cur_h = enc_h
    cur_c = enc_c
    generated = []

    for _ in range(max_len):
        # эмбеддинг для текущего шага (batch=1, timesteps=1)
        dec_emb_step = embedding(cur_token)  # (1,1,units)

        # прогоним LSTM один шаг, передав текущие состояния
        dec_out_step, h_new, c_new = decoder_lstm(dec_emb_step, initial_state=[cur_h, cur_c])
        # dec_out_step shape: (1,1,units)

        # attention: query = dec_out_step, value = enc_out
        context_step = attn_layer([dec_out_step, enc_out])  # (1,1,units)

        concat_step = tf.concat([dec_out_step, context_step], axis=-1)  # (1,1,2*units)

        # logits для этого шага (вызов TimeDistributed(Dense) на единичном шаге)
        logits_step = logits_layer(concat_step)  # (1,1,vocab)
        logits_step = tf.squeeze(logits_step, axis=1)  # (1, vocab)
        next_id = int(tf.argmax(logits_step, axis=-1).numpy()[0])

        if next_id == tokenizer.eos_token_id:
            break
        generated.append(next_id)

        # подготовка к следующему шагу
        cur_token = np.array([[next_id]], dtype=np.int32)
        cur_h, cur_c = h_new, c_new

    # раскодируем
    return tokenizer.decode(generated, clean_up_tokenization_spaces=True)

In [26]:
print(translate("Я работаю очень много", tokenizer, model))
print(translate("Это новая фотография?", tokenizer, model))

I'm not going to be a doctor.
It's a very long time.


In [27]:
print(translate("Моему учителю физики все равно, пропущу ли я занятия", tokenizer, model))

My father is a good teacher, but I'm not going to do that.
